# AgroSele — Modelo Híbrido (BM25 + BERTimbau + MLP)

Quarta variante, e a mais ambiciosa: em vez de tratar PLN clássico e
BERTimbau como duas abordagens concorrentes (como fiz nos notebooks
anteriores), aqui eu **fundo os dois sinais num vetor só**, antes de jogar
pro MLP.

A pergunta que motivou isso: o TF-IDF (PLN clássico) ganha do BERTimbau cru
porque captura correspondência lexical exata (nome de doença, insumo,
procedimento) que o embedding genérico "borra". E se, em vez de escolher um
ou outro, eu desse pro MLP os dois sinais juntos — o vetor ddenso do BERT
**e** um escore lexical clássico (BM25)? Será que ele recupera parte do
ganho que normalmente só vem de fazer fine-tuning (que custa ~11h de
treino)?

Arquitetura (4 camadas):

```
1. PLN classico       -> indice BM25 (rank_bm25) sobre o corpus de respostas
2. Matematica do BERT -> cosseno, |diferenca|, produto (funcoes explicitas)
3. Feature Fusion     -> concatena os dois sinais num vetor so (3074d)
4. MLP (PyTorch)      -> Linear -> ReLU -> Dropout -> ... -> Sigmoid (saida explicita)
```

In [1]:
import csv
import itertools
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import load_dataset
from rank_bm25 import BM25Okapi

SEMENTE = 42
random.seed(SEMENTE)
np.random.seed(SEMENTE)

C:\Users\frede\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Camada 1 — PLN clássico (BM25)

Uso a biblioteca `rank_bm25` (permitida no enunciado, junto com scikit-learn)
em vez de reimplementar BM25 na mão — já fiz TF-IDF do zero no outro
notebook, aqui o foco é a arquitetura híbrida, não reprovar a fórmula do
BM25 de novo.

Tokenização e stopwords são as mesmas do notebook clássico.

In [2]:
LISTA_STOPWORDS_PT = set("""
a ao aos aquela aquelas aquele aqueles aquilo as até com como da das de dele
deles depois do dos e ela elas ele eles em entre era eram essa essas esse
esses esta estamos estas estava estavam este esteja estejam estejamos estes
esteve estive estivemos estiveram estivesse estivessem estivéramos
estivéssemos estou está estávamos estão eu foi fomos for fora foram forem
formos fosse fossem fui fôramos fôssemos haja hajam hajamos havemos hei
houve houvemos houver houvera houveram houverei houverem houveremos
houveria houveriam houvermos houverá houverão houveríamos houvesse
houvessem houvéramos houvéssemos há hão isso isto já lhe lhes mais mas me
mesmo meu meus minha minhas muito na nas nem no nos nossa nossas nosso
nossos num numa não nós o os ou para pela pelas pelo pelos por qual quando
que quem se seja sejam sejamos sem serei seremos seria seriam será serão
seríamos seu seus somos sou sua suas são só também te tem temos tenha
tenham tenhamos tenho terei teremos teria teriam terá terão teríamos teu
teus teve tinha tinham tive tivemos tiver tivera tiveram tiverem tivermos
tivesse tivessem tivéramos tivéssemos tu tua tuas tá um uma você vocês
vos à às éramos é
""".split())

import re
padrao_token = re.compile(r"[a-zà-öø-ÿ0-9]+")


def tokenizar(texto):
    palavras = padrao_token.findall(texto.lower())
    return [p for p in palavras if p not in LISTA_STOPWORDS_PT and len(p) > 1]


class CamadaBM25:
    """Indice BM25 (rank_bm25.BM25Okapi) sobre o corpus de respostas.

    O IDF e o comprimento medio de documento sao ajustados uma unica vez
    sobre o corpus inteiro (2657 respostas). Os escores brutos ficam em
    cache por pergunta, pra nao recalcular contra o corpus inteiro toda
    vez que o mesmo par (pergunta, candidata) aparece nos pares de treino
    ou na avaliacao.
    """

    def __init__(self, ids_documentos, textos_documentos):
        self.ids_documentos = list(ids_documentos)
        self.indice_do_id = {id_doc: i for i, id_doc in enumerate(self.ids_documentos)}
        corpus_tokenizado = [tokenizar(t) for t in textos_documentos]
        self.bm25 = BM25Okapi(corpus_tokenizado)
        self._cache_escores_brutos = {}

    def _escores_brutos(self, id_pergunta, texto_pergunta):
        if id_pergunta not in self._cache_escores_brutos:
            self._cache_escores_brutos[id_pergunta] = np.asarray(
                self.bm25.get_scores(tokenizar(texto_pergunta)), dtype=np.float64)
        return self._cache_escores_brutos[id_pergunta]

    def escores_normalizados_do_pool(self, id_pergunta, texto_pergunta, ids_candidatas):
        """Escore BM25 normalizado (min-max, entre 0 e 1) so entre as
        candidatas de UM pool (as ate 50 candidatas daquela pergunta) --
        e essa comparacao que importa pro ranking, e evita que a escala
        nao-limitada do BM25 bruto atrapalhe o treino do MLP."""
        brutos = self._escores_brutos(id_pergunta, texto_pergunta)
        indices = [self.indice_do_id[c] for c in ids_candidatas]
        escores_pool = brutos[indices]
        minimo, maximo = escores_pool.min(), escores_pool.max()
        amplitude = (maximo - minimo) or 1.0
        normalizados = (escores_pool - minimo) / amplitude
        return {c: float(s) for c, s in zip(ids_candidatas, normalizados)}

## 2. Camada 2 — Matemática do BERT, explícita

Em vez de escrever tudo numa linha só (`torch.cat([q, a, |q-a|, q*a])`),
separo cada operação em uma função própria, até pra deixar claro o que cada
pedaço do vetor final representa.

In [3]:
def similaridade_cosseno(u, v, eps=1e-8):
    """cos(u, v) = (u . v) / (||u|| * ||v||), calculado por extenso -- nao
    uso torch.nn.functional.cosine_similarity de proposito, pra deixar cada
    termo da formula visivel. Funciona tanto pra um par (768,) quanto pra
    um lote (N, 768)."""
    produto_escalar = (u * v).sum(dim=-1)
    norma_u = torch.sqrt((u * u).sum(dim=-1))
    norma_v = torch.sqrt((v * v).sum(dim=-1))
    return produto_escalar / (norma_u * norma_v + eps)


def diferenca_absoluta(u, v):
    """|u - v|, elemento a elemento -- mantem a dimensionalidade (768,)."""
    return torch.abs(u - v)


def produto_elemento_a_elemento(u, v):
    """u * v, elemento a elemento -- mantem a dimensionalidade (768,)."""
    return u * v


def features_do_par(emb_pergunta, emb_resposta):
    """Vetor denso [pergunta, resposta, |diferenca|, produto] -- 4 blocos
    de 768 = 3072 dimensoes. Isso e o vetor "so BERT", antes de fundir com
    o PLN classico (proxima secao)."""
    diferenca = diferenca_absoluta(emb_pergunta, emb_resposta)
    produto = produto_elemento_a_elemento(emb_pergunta, emb_resposta)
    return torch.cat([emb_pergunta, emb_resposta, diferenca, produto], dim=-1)


# teste rapido pra conferir as dimensoes
u_teste = torch.randn(768)
v_teste = torch.randn(768)
print("cosseno:", similaridade_cosseno(u_teste, v_teste).item())
print("features_do_par:", features_do_par(u_teste, v_teste).shape)

cosseno: -0.03453539311885834
features_do_par: torch.Size([3072])


## 3. Camada 3 — Feature Fusion

Aqui é o núcleo da ideia: pego o vetor denso do BERT (3072d) e concateno
dois escores extras — a similaridade de cosseno do BERT (semântico) e o
escore BM25 normalizado (lexical, do PLN clássico) — formando um vetor
único e mais rico, de 3074 dimensões.

In [4]:
def fundir_features(emb_pergunta, emb_resposta, escore_bm25):
    """Enriquece o vetor de pares do BERT (3072d) concatenando dois escores
    escalares: a similaridade de cosseno explicita entre os embeddings, e o
    escore BM25 normalizado da camada classica. Resultado: vetor unico de
    3074 dimensoes. Funciona tanto pra um unico par quanto pra um lote
    (usado na avaliacao por ranking)."""
    vetor_denso = features_do_par(emb_pergunta, emb_resposta)         # (..., 3072)
    cosseno = similaridade_cosseno(emb_pergunta, emb_resposta)        # (...,)
    extras = torch.stack([cosseno, escore_bm25], dim=-1)              # (..., 2)
    return torch.cat([vetor_denso, extras], dim=-1)                   # (..., 3074)


# teste rapido
vetor_fundido = fundir_features(u_teste, v_teste, torch.tensor(0.42))
print("vetor enriquecido:", vetor_fundido.shape)

vetor enriquecido: torch.Size([3074])


## 4. Camada 4 — Rede Neural (MLP) em PyTorch

Camadas lineares explícitas, intercaladas com ReLU e Dropout, e no final um
`nn.Sigmoid()` **explícito** — não uso `BCEWithLogitsLoss` (que já embute o
sigmoid), uso `nn.Sigmoid()` dentro do modelo mesmo e treino com
`nn.BCELoss()` sobre a probabilidade já calculada.

In [5]:
class MLPHibrido(nn.Module):
    """MLP para classificacao binaria (match / nao-match) do par
    (pergunta, candidata), recebendo o vetor enriquecido (BERT + PLN
    classico). A saida passa por Sigmoid explicito, entao ja sai como
    probabilidade em [0, 1]."""

    def __init__(self, dim_entrada, ocultas, dropout):
        super().__init__()
        self.rede = nn.Sequential(
            nn.Linear(dim_entrada, ocultas),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ocultas, ocultas // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ocultas // 2, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.rede(x).squeeze(-1)

## 5. Montagem de pares e avaliação (com a fusão aplicada)

In [6]:
N_NEGATIVOS_TREINO = 8
FRACAO_NEGATIVOS_DIFICEIS = 0.5


def montar_pares(conjunto, textos_perguntas, emb_perguntas, emb_respostas, bm25, n_negativos, fracao_dificeis=0.0):
    linhas = list(conjunto)
    n_dificeis = int(round(n_negativos * fracao_dificeis))
    n_aleatorios = n_negativos - n_dificeis

    X, y = [], []
    for linha in linhas:
        id_pergunta = linha["query-id"]
        id_certa = linha["positive-doc-id"]
        pool = linha["candidates-ids"]
        candidatas = [c for c in pool if c != id_certa]

        if n_dificeis > 0 and len(candidatas) > n_dificeis:
            pergunta_norm = torch.nn.functional.normalize(emb_perguntas[id_pergunta].unsqueeze(0), dim=-1)
            candidatas_norm = torch.nn.functional.normalize(
                torch.stack([emb_respostas[c] for c in candidatas]), dim=-1)
            similaridades = (candidatas_norm @ pergunta_norm.T).squeeze(-1)
            indices_dificeis = torch.argsort(-similaridades)[:n_dificeis].tolist()
            negativos_dificeis = [candidatas[i] for i in indices_dificeis]
            restantes = [c for c in candidatas if c not in negativos_dificeis]
            negativos_aleatorios = random.sample(restantes, min(n_aleatorios, len(restantes)))
            negativos = negativos_dificeis + negativos_aleatorios
        else:
            negativos = random.sample(candidatas, min(n_negativos, len(candidatas)))

        texto_pergunta = textos_perguntas[id_pergunta]
        escores_bm25 = bm25.escores_normalizados_do_pool(id_pergunta, texto_pergunta, pool)
        emb_p = emb_perguntas[id_pergunta]

        X.append(fundir_features(emb_p, emb_respostas[id_certa], torch.tensor(escores_bm25[id_certa])))
        y.append(1)
        for id_neg in negativos:
            X.append(fundir_features(emb_p, emb_respostas[id_neg], torch.tensor(escores_bm25[id_neg])))
            y.append(0)

    return torch.stack(X), torch.tensor(y, dtype=torch.float32)


def avaliar_ranking(modelo, conjunto, textos_perguntas, emb_perguntas, emb_respostas, bm25):
    modelo.eval()
    lista_acuracia1, lista_mrr = [], []
    with torch.no_grad():
        for linha in conjunto:
            id_pergunta, id_certa, candidatas = linha["query-id"], linha["positive-doc-id"], linha["candidates-ids"]
            texto_pergunta = textos_perguntas[id_pergunta]
            escores_bm25 = bm25.escores_normalizados_do_pool(id_pergunta, texto_pergunta, candidatas)

            pergunta = emb_perguntas[id_pergunta].unsqueeze(0).expand(len(candidatas), -1)
            respostas = torch.stack([emb_respostas[c] for c in candidatas])
            vetor_bm25 = torch.tensor([escores_bm25[c] for c in candidatas], dtype=torch.float32)

            features = fundir_features(pergunta, respostas, vetor_bm25)
            pontuacoes = modelo(features).numpy()  # ja e probabilidade (Sigmoid interno)

            ordem = np.argsort(-pontuacoes)
            ids_ranqueados = [candidatas[i] for i in ordem]
            posicao = ids_ranqueados.index(id_certa) + 1

            lista_acuracia1.append(1.0 if posicao == 1 else 0.0)
            lista_mrr.append(1.0 / posicao)
    return float(np.mean(lista_acuracia1)), float(np.mean(lista_mrr))


def treinar_um_modelo(X_treino, y_treino, conjunto_dev, textos_perguntas, emb_perguntas,
                       emb_respostas, bm25, ocultas, dropout, taxa_aprendizado):
    torch.manual_seed(SEMENTE)
    modelo = MLPHibrido(X_treino.shape[1], ocultas, dropout)
    otimizador = torch.optim.Adam(modelo.parameters(), lr=taxa_aprendizado)
    funcao_perda = nn.BCELoss()  # o modelo ja devolve probabilidade (Sigmoid interno)

    melhor_mrr, melhor_estado, sem_melhora = -1.0, None, 0
    n = X_treino.shape[0]
    tamanho_lote = 64
    gerador = torch.Generator().manual_seed(SEMENTE)

    for epoca in range(25):
        modelo.train()
        permutacao = torch.randperm(n, generator=gerador)
        for i in range(0, n, tamanho_lote):
            indices = permutacao[i:i + tamanho_lote]
            otimizador.zero_grad()
            probabilidades = modelo(X_treino[indices])
            perda = funcao_perda(probabilidades, y_treino[indices])
            perda.backward()
            otimizador.step()

        _, mrr_dev = avaliar_ranking(modelo, conjunto_dev, textos_perguntas, emb_perguntas, emb_respostas, bm25)
        if mrr_dev > melhor_mrr:
            melhor_mrr, melhor_estado, sem_melhora = mrr_dev, {k: v.clone() for k, v in modelo.state_dict().items()}, 0
        else:
            sem_melhora += 1
            if sem_melhora >= 4:  # paciencia
                break

    modelo.load_state_dict(melhor_estado)
    return modelo, melhor_mrr

## 6. Rodando tudo

Reaproveito os embeddings do BERTimbau já cacheados na pasta do projeto
base (congelados, sem fine-tuning) — o ganho aqui vem inteiramente da fusão
com o BM25, não de ajustar os pesos do BERT.

In [7]:
print("Carregando embeddings BERTimbau cacheados (congelados)...")
emb_perguntas = torch.load("../selecao-resposta-milkqa/cache/queries_embeddings.pt", weights_only=False)
emb_respostas = torch.load("../selecao-resposta-milkqa/cache/corpus_embeddings.pt", weights_only=False)
print(f"{len(emb_perguntas)} perguntas | {len(emb_respostas)} respostas | dim={next(iter(emb_perguntas.values())).shape[0]}")


def carregar_csv(caminho):
    linhas = {}
    with open(caminho, encoding="utf-8") as f:
        for linha in csv.DictReader(f):
            linhas[linha["id"]] = linha["text"]
    return linhas


textos_respostas = carregar_csv("../selecao-resposta-milkqa/datasets/corpus.csv")
textos_perguntas = carregar_csv("../selecao-resposta-milkqa/datasets/queries.csv")

print("Construindo indice BM25 sobre o corpus de respostas...")
bm25 = CamadaBM25(list(textos_respostas.keys()), list(textos_respostas.values()))

print("Carregando splits oficiais do MilkQA...")
ds = load_dataset("eduagarcia/MilkQA")
conjunto_treino, conjunto_dev, conjunto_teste = ds["train"], ds["dev"], ds["test"]
print(f"treino={len(conjunto_treino)} | dev={len(conjunto_dev)} | teste={len(conjunto_teste)}")

Carregando embeddings BERTimbau cacheados (congelados)...


2657 perguntas | 2657 respostas | dim=768
Construindo indice BM25 sobre o corpus de respostas...


Carregando splits oficiais do MilkQA...


treino=2307 | dev=50 | teste=300


## Verificação: o cache de embeddings é fiel ao BERT de verdade?

Os `emb_perguntas`/`emb_respostas` acima vieram de um cache em disco
(gerado no notebook do bi-encoder congelado), não foram recalculados
agora. Confiro abaixo, ao vivo, se 5 embeddings amostrados batem com o que
o BERTimbau produz de fato — o modelo está congelado, então recalcular tem
que dar exatamente o mesmo vetor.

In [8]:
import random as _random

print("Verificando fidelidade do cache: recalculando 5 exemplos ao vivo...")
from transformers import AutoModel, AutoTokenizer

_ids_amostra = _random.Random(123).sample(list(emb_perguntas.keys()), 5)

def _pooling_media(ultima_camada, mascara):
    m = mascara.unsqueeze(-1).expand(ultima_camada.size()).float()
    return torch.sum(ultima_camada * m, dim=1) / torch.clamp(m.sum(dim=1), min=1e-9)

_tok_verif = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")
_modelo_verif = AutoModel.from_pretrained("neuralmind/bert-base-portuguese-cased")
_modelo_verif.eval()

_diffs = []
with torch.no_grad():
    for _id in _ids_amostra:
        _entrada = _tok_verif([textos_perguntas[_id]], return_tensors="pt",
                               truncation=True, max_length=256, padding=True)
        _saida = _modelo_verif(**_entrada)
        _vetor_ao_vivo = _pooling_media(_saida.last_hidden_state, _entrada["attention_mask"])[0]
        _vetor_cache = emb_perguntas[_id]
        _diff = (_vetor_ao_vivo - _vetor_cache).abs().max().item()
        _diffs.append(_diff)
        print(f"  pergunta {_id}: diferenca maxima entre cache e recalculo ao vivo = {_diff:.2e}")

assert max(_diffs) < 1e-4, "cache NAO bate com o recalculo ao vivo do BERT!"
print("Cache confirmado: os vetores salvos batem com o recalculo ao vivo do BERTimbau.")
del _modelo_verif, _tok_verif

Verificando fidelidade do cache: recalculando 5 exemplos ao vivo...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13422.53it/s]


[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  pergunta 2444: diferenca maxima entre cache e recalculo ao vivo = 2.38e-07
  pergunta 9969: diferenca maxima entre cache e recalculo ao vivo = 2.38e-07


  pergunta 3679: diferenca maxima entre cache e recalculo ao vivo = 2.68e-07
  pergunta 14273: diferenca maxima entre cache e recalculo ao vivo = 2.09e-07


  pergunta 9938: diferenca maxima entre cache e recalculo ao vivo = 2.25e-07
Cache confirmado: os vetores salvos batem com o recalculo ao vivo do BERTimbau.


In [9]:
print(f"\nMontando pares de treino (todas as {len(conjunto_treino)} perguntas, "
      f"{N_NEGATIVOS_TREINO} negativos cada, vetor enriquecido BERT+BM25)...")
X_treino, y_treino = montar_pares(conjunto_treino, textos_perguntas, emb_perguntas, emb_respostas,
                                   bm25, N_NEGATIVOS_TREINO, fracao_dificeis=FRACAO_NEGATIVOS_DIFICEIS)
print(f"{X_treino.shape[0]} pares ({int(y_treino.sum())} positivos, {int((1 - y_treino).sum())} negativos) "
      f"| dim do vetor enriquecido = {X_treino.shape[1]}")


Montando pares de treino (todas as 2307 perguntas, 8 negativos cada, vetor enriquecido BERT+BM25)...


20763 pares (2307 positivos, 18456 negativos) | dim do vetor enriquecido = 3074


In [10]:
print("\n===== Grid Search (selecao pelo MRR no dev) =====")
grade = {
    "ocultas": [128, 256],
    "dropout": [0.2, 0.4],
    "taxa_aprendizado": [1e-3, 1e-4],
}
combinacoes = list(itertools.product(grade["ocultas"], grade["dropout"], grade["taxa_aprendizado"]))
resultados = []
melhor_geral = {"mrr": -1.0, "modelo": None, "config": None}

for ocultas, dropout, taxa_aprendizado in combinacoes:
    modelo, mrr_dev = treinar_um_modelo(X_treino, y_treino, conjunto_dev, textos_perguntas,
                                         emb_perguntas, emb_respostas, bm25, ocultas, dropout, taxa_aprendizado)
    acuracia1_dev, _ = avaliar_ranking(modelo, conjunto_dev, textos_perguntas, emb_perguntas, emb_respostas, bm25)
    resultados.append({"ocultas": ocultas, "dropout": dropout, "taxa_aprendizado": taxa_aprendizado,
                        "mrr_dev": mrr_dev, "acuracia1_dev": acuracia1_dev})
    print(f"  ocultas={ocultas:4d} dropout={dropout:.1f} lr={taxa_aprendizado:.0e} "
          f"-> dev MRR={mrr_dev:.4f} Acc@1={acuracia1_dev:.4f}")
    if mrr_dev > melhor_geral["mrr"]:
        melhor_geral = {"mrr": mrr_dev, "modelo": modelo, "config": (ocultas, dropout, taxa_aprendizado)}

ocultas, dropout, taxa_aprendizado = melhor_geral["config"]
print(f"\nMelhor configuracao: ocultas={ocultas}, dropout={dropout}, lr={taxa_aprendizado} "
      f"(dev MRR={melhor_geral['mrr']:.4f})")


===== Grid Search (selecao pelo MRR no dev) =====


  ocultas= 128 dropout=0.2 lr=1e-03 -> dev MRR=0.7932 Acc@1=0.7200


  ocultas= 128 dropout=0.2 lr=1e-04 -> dev MRR=0.7626 Acc@1=0.6800


  ocultas= 128 dropout=0.4 lr=1e-03 -> dev MRR=0.7949 Acc@1=0.7000


  ocultas= 128 dropout=0.4 lr=1e-04 -> dev MRR=0.6309 Acc@1=0.5400


  ocultas= 256 dropout=0.2 lr=1e-03 -> dev MRR=0.7925 Acc@1=0.7000


  ocultas= 256 dropout=0.2 lr=1e-04 -> dev MRR=0.7884 Acc@1=0.7000


  ocultas= 256 dropout=0.4 lr=1e-03 -> dev MRR=0.7996 Acc@1=0.7200


  ocultas= 256 dropout=0.4 lr=1e-04 -> dev MRR=0.7762 Acc@1=0.6800

Melhor configuracao: ocultas=256, dropout=0.4, lr=0.001 (dev MRR=0.7996)


In [11]:
print("\n===== Avaliacao final no TESTE (300 perguntas, 50 candidatas cada) =====")
modelo_final = melhor_geral["modelo"]
acuracia1_teste, mrr_teste = avaliar_ranking(modelo_final, conjunto_teste, textos_perguntas,
                                              emb_perguntas, emb_respostas, bm25)
print(f"Accuracy@1 (teste, hibrido BERT+BM25+MLP) = {acuracia1_teste:.4f}")
print(f"MRR (teste, hibrido BERT+BM25+MLP)        = {mrr_teste:.4f}")
print("\nComparar com: congelado sem BM25 = 0.570/0.679 | fine-tuning completo = 0.690/0.782")


===== Avaliacao final no TESTE (300 perguntas, 50 candidatas cada) =====


Accuracy@1 (teste, hibrido BERT+BM25+MLP) = 0.6633
MRR (teste, hibrido BERT+BM25+MLP)        = 0.7534

Comparar com: congelado sem BM25 = 0.570/0.679 | fine-tuning completo = 0.690/0.782


In [12]:
import os
os.makedirs("checkpoints", exist_ok=True)
torch.save({
    "model_state": modelo_final.state_dict(),
    "ocultas": ocultas, "dropout": dropout, "taxa_aprendizado": taxa_aprendizado,
    "dim_entrada": X_treino.shape[1],
    "acuracia1_teste": acuracia1_teste, "mrr_teste": mrr_teste,
}, "checkpoints/best_model_hibrido_notebook.pt")
pd.DataFrame(resultados).sort_values("mrr_dev", ascending=False).to_csv(
    "checkpoints/grid_search_results_notebook.csv", index=False)
print("Checkpoint e grid search salvos em checkpoints/")

Checkpoint e grid search salvos em checkpoints/


## Conclusão

O híbrido (BERTimbau congelado + BM25 fundido no vetor de entrada) chega em
`Accuracy@1 ≈ 0,66` e `MRR ≈ 0,75` — bem acima da versão só-congelada
(0,570/0,679) e quase alcançando o fine-tuning completo (0,690/0,782),
**sem ajustar um único peso do BERT**, a um custo de treino de minutos em
vez de ~11 horas.

Isso sugere que boa parte do ganho que normalmente se atribui ao
fine-tuning não vem de uma capacidade representacional nova que o BERT
"aprende" — vem de disponibilizar, de alguma forma, o sinal de
correspondência lexical exata que o embedding congelado por si só não
expõe de forma acessível ao classificador. Fundir esse sinal explicitamente
(via BM25) é uma forma bem mais barata de chegar perto do mesmo lugar.